In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


merged_df = pd.read_parquet('/home/lkapral/hb/data/resampled/hourly_fused.parquet')



plt.figure(figsize=(10, 5))
sns.histplot(merged_df['blood_pressure_systolic_mmHg'].dropna(), bins=30, kde=True, color='blue')
plt.title(f'Distribution of blood_pressure_systolic_mmHg', fontsize=14)
plt.xlabel('blood_pressure_systolic_mmHg', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.show()

merged_df.sort_values(by=['encounterId', 'utcChartTime'], inplace=True)

# Remove IDs where there are less than 4 steps
merged_df = merged_df.groupby('encounterId').filter(lambda x: len(x) >= 4)

# Remove IDs where there is no single entry in column hemoglobin_g/dl
merged_df = merged_df[merged_df.groupby('encounterId')['hemoglobin_g/dl'].transform('count') > 0]

merged_df['sex_or_gender'].replace({'männlich': 'M', 'weiblich': 'F'}, inplace=True)
merged_df['sex_or_gender'].replace({'männlich': 'M', 'W': 'F'}, inplace=True)


# Check library versions
print(f"Pandas version: {pd.__version__}")
print(f"Seaborn version: {sns.__version__}")




In [ ]:
import numpy as np
import pandas as pd


# Conversion factors (to norepinephrine equivalents mcg/kg/min)
VASO_CONVERSION_FACTORS = {
    'norepinephrine': 1.0,    # Reference drug (mcg/kg/min)
    'epinephrine': 1.0,       # Same potency as norepinephrine
    'vasopressin': 5.0,       # U/h or IE/h -> NE equivalent
    'phenylephrine': 0.45,    # Less potent
    'dopamine': 0.01,         # Much less potent as vasopressor
    'dobutamine': 0.01,       # Primarily inotrope, weak vasopressor
}


# Preset column mappings for different datasets
VASO_COLUMN_PRESETS = {
    'mimic': {
        'norepinephrine': 'avg_norepinephrine',
        'epinephrine': 'avg_epinephrine',
        'vasopressin': 'avg_vasopressin',
        'phenylephrine': 'avg_phenylephrine',
        'dopamine': 'avg_dobutamine',
    },
    'muw': {
        'norepinephrine': 'norepinephrine_µg/kg/min',
        'vasopressin': 'vasopressin_IE/h',
        'dobutamine': 'dobutamine_µg/kg/min',
    },
    'mimic_raw': {
        'norepinephrine': 'norepinephrine_dose',
        'epinephrine': 'epinephrine_dose',
        'vasopressin': 'vasopressin_dose',
        'phenylephrine': 'phenylephrine_dose',
        'dopamine': 'dopamine_dose',
    },
}


def calculate_combined_vaso(
    df,
    preset=None,
    column_mapping=None,
    output_col='combined_vaso',
    inplace=False,
    verbose=True
):
    """
    Calculate combined vasopressor dose in norepinephrine equivalents (mcg/kg/min).
    
    Conversion factors based on MIMIC extraction code:
    - Norepinephrine: 1.0 (reference)
    - Epinephrine: 1.0 (same potency as norepinephrine)
    - Vasopressin: 5.0 (from U/h or IE/h to NE equivalent)
    - Phenylephrine: 0.45 (less potent)
    - Dopamine/Dobutamine: 0.01 (much less potent as vasopressor)
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing vasopressor columns
    preset : str, optional
        Use a preset column mapping: 'mimic', 'muw', or 'mimic_raw'
    column_mapping : dict, optional
        Custom mapping of drug names to column names.
        Keys: 'norepinephrine', 'epinephrine', 'vasopressin', 'phenylephrine', 'dopamine', 'dobutamine'
        Values: column names in df
    output_col : str
        Name of the output column for combined vaso
    inplace : bool
        If True, modify df in place. If False, return a copy.
    verbose : bool
        If True, print summary statistics.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with combined_vaso column added
    
    Examples
    --------
    >>> # Using preset for MIMIC
    >>> df = calculate_combined_vaso(df, preset='mimic')
    
    >>> # Using preset for MUW
    >>> df_muw = calculate_combined_vaso(df_muw, preset='muw')
    
    >>> # Custom column mapping
    >>> df = calculate_combined_vaso(df, column_mapping={
    ...     'norepinephrine': 'norepi_dose',
    ...     'vasopressin': 'vaso_dose'
    ... })
    """
    
    if not inplace:
        df = df.copy()
    
    # Determine column mapping
    if preset is not None:
        if preset not in VASO_COLUMN_PRESETS:
            raise ValueError(f"Unknown preset '{preset}'. Available: {list(VASO_COLUMN_PRESETS.keys())}")
        col_map = VASO_COLUMN_PRESETS[preset].copy()
    elif column_mapping is not None:
        col_map = column_mapping.copy()
    else:
        # Try to auto-detect columns
        col_map = _auto_detect_vaso_columns(df)
        if verbose:
            print(f"Auto-detected vaso columns: {col_map}")
    
    # Initialize combined vaso
    df[output_col] = 0.0
    
    # Track which columns exist and were used
    cols_used = []
    cols_missing = []
    
    # Sum up all vasopressors with their conversion factors
    for drug, factor in VASO_CONVERSION_FACTORS.items():
        col = col_map.get(drug)
        if col is not None and col in df.columns:
            df[output_col] += df[col].fillna(0) * factor
            cols_used.append((drug, col, factor))
        elif col is not None:
            cols_missing.append((drug, col))
    
    # Set to NaN if ALL vaso columns were NaN (no data, not zero dose)
    if cols_used:
        used_cols = [col for _, col, _ in cols_used]
        all_nan_mask = pd.concat([df[col].isna() for col in used_cols], axis=1).all(axis=1)
        df.loc[all_nan_mask, output_col] = np.nan
    
    # Print summary
    if verbose:
        print(f"\n=== Combined Vasopressor Calculation ===")
        print(f"Output column: '{output_col}'")
        print(f"\nColumns used:")
        for drug, col, factor in cols_used:
            non_zero = (df[col].fillna(0) > 0).sum()
            print(f"  {drug} ({col}) x {factor}: {non_zero} non-zero values")
        if cols_missing:
            print(f"\nColumns not found (skipped):")
            for drug, col in cols_missing:
                print(f"  {drug} ({col})")
        print(f"\nResult:")
        print(f"  Non-null: {df[output_col].notna().sum()} / {len(df)}")
        print(f"  Non-zero: {(df[output_col].fillna(0) > 0).sum()}")
        print(f"  Mean (when > 0): {df.loc[df[output_col] > 0, output_col].mean():.4f}")
        print(f"  Median (when > 0): {df.loc[df[output_col] > 0, output_col].median():.4f}")
        print(f"  Max: {df[output_col].max():.4f}")
    
    return df


def _auto_detect_vaso_columns(df):
    """
    Auto-detect vasopressor columns based on common naming patterns.
    """
    col_map = {}
    
    patterns = {
        'norepinephrine': ['norepinephrine', 'norepi', 'norad', 'levophed'],
        'epinephrine': ['epinephrine', 'epi', 'adrenalin'],
        'vasopressin': ['vasopressin', 'vaso_', 'avp'],
        'phenylephrine': ['phenylephrine', 'phenyl', 'neosynephrine'],
        'dopamine': ['dopamine'],
        'dobutamine': ['dobutamine'],
    }
    
    for drug, keywords in patterns.items():
        for col in df.columns:
            col_lower = col.lower()
            if any(kw in col_lower for kw in keywords):
                col_map[drug] = col
                break
    
    return col_map


def calculate_combined_vaso_simple(
    norepinephrine=None,
    epinephrine=None,
    vasopressin=None,
    phenylephrine=None,
    dopamine=None,
    dobutamine=None
):
    """
    Calculate combined vaso from individual values (for single calculations).
    
    Parameters
    ----------
    norepinephrine : float
        Norepinephrine dose (mcg/kg/min)
    epinephrine : float
        Epinephrine dose (mcg/kg/min)
    vasopressin : float
        Vasopressin dose (U/h or IE/h)
    phenylephrine : float
        Phenylephrine dose (mcg/kg/min)
    dopamine : float
        Dopamine dose (mcg/kg/min)
    dobutamine : float
        Dobutamine dose (mcg/kg/min)
    
    Returns
    -------
    float
        Combined vasopressor dose in norepinephrine equivalents (mcg/kg/min)
    
    Example
    -------
    >>> combined = calculate_combined_vaso_simple(norepinephrine=0.1, vasopressin=0.04)
    """
    
    values = {
        'norepinephrine': norepinephrine,
        'epinephrine': epinephrine,
        'vasopressin': vasopressin,
        'phenylephrine': phenylephrine,
        'dopamine': dopamine,
        'dobutamine': dobutamine,
    }
    
    total = 0.0
    has_any = False
    
    for drug, value in values.items():
        if value is not None and not np.isnan(value):
            total += value * VASO_CONVERSION_FACTORS.get(drug, 0)
            has_any = True
    
    return total if has_any else np.nan

merged_df = calculate_combined_vaso(merged_df, preset='muw')

In [ ]:
merged_df

In [ ]:
# -------------------- Data Cleaning --------------------

# 1. Convert 'utcChartTime' to datetime
merged_df['utcChartTime'] = pd.to_datetime(merged_df['utcChartTime'], errors='coerce')

# 2. Convert numeric columns to appropriate types
numeric_columns = [
    'blood_pressure_systolic_mmHg', 'blood_pressure_mean_mmHg',
    'blood_pressure_diastolic_mmHg', 'norepinephrine_µg/kg/min',
    'dobutamine_µg/kg/min', 'vasopressin_IE/h', 'fluids_ml', 'colloids_ml',
    'fibrinogen_mg/dl', 'platelet_count_G/l',
    'lactate_mmol/l', 'base_excess_mmol/l', 'hemoglobin_g/dl',
    'harnk_ml', 'thorax-drain_ml', 'jackson-pratt_ml', 'redon-drain_ml',
    'easy-flow_ml', 'robinson-drain_ml', 'age', 'heart_rate',
    'respiratory_rate', 'spo2', 'blood_input', 'combined_vaso',
    'urine_output_ml'
]

for col in numeric_columns:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

# 4. Handle missing values
# Drop rows with missing 'encounterId' or 'utcChartTime'
merged_df = merged_df.dropna(subset=['encounterId', 'utcChartTime'])

# 5a. ARTIFACT RANGES: values outside these are set to NaN (assumed measurement error)

artifact_ranges = {
    'norepinephrine_µg/kg/min': (0.0, 5.0),       # >5 µg/kg/min very likely documentation / unit error
    'dobutamine_µg/kg/min': (0.0, 40.0),          # >40 µg/kg/min very unlikely
    'vasopressin_IE/h': (0.0, 10.0),              # >10 IE/h very unlikely
    'fluids_ml': (0.0, 200000.0),                 # extremely high to catch absurd values
    'colloids_ml': (0.0, 20000.0),
    'fibrinogen_mg/dl': (0.0, 4000.0),
    'platelet_count_G/l': (0.0, 2500.0),
    'lactate_mmol/l': (0.0, 50.0),
    'base_excess_mmol/l': (-50.0, 50.0),
    'urine_output_ml': (0.0, 25000.0),
    'thorax-drain_ml': (0.0, 20000.0),
    'jackson-pratt_ml': (0.0, 20000.0),
    'redon-drain_ml': (0.0, 20000.0),
    'easy-flow_ml': (0.0, 20000.0),
    'robinson-drain_ml': (0.0, 20000.0),
    'harnk_ml': (0.0, 20000.0),
    'age': (0.0, 130.0),                          # hard artifacts (negatives, ultra-high)
    'heart_rate': (0.0, 350.0),
    'respiratory_rate': (0.0, 200.0),
    'blood_input': (0.0, 20000.0),
    'spo2': (0.0, 110.0),
    'combined_vaso' : (0.0, 20.0),

    'blood_pressure_systolic_mmHg': (20.0, 350.0),
    'blood_pressure_diastolic_mmHg': (10.0, 250.0),
    'blood_pressure_mean_mmHg': (20.0, 300.0),

    'hemoglobin_g/dl': (2.0, 30.0)
}

for col, (min_art, max_art) in artifact_ranges.items():
    if col in merged_df.columns:
        merged_df.loc[(merged_df[col] < min_art) | (merged_df[col] > max_art), col] = np.nan
        # Optional: keep an artifact flag instead of just NaN
        # merged_df[col + '_artifact'] = ((merged_df[col] < min_art) | (merged_df[col] > max_art)).astype(int)

# 5b. CLAMPING RANGES: for the remaining values, clamp into medically meaningful bounds


clamp_ranges = {
    'norepinephrine_µg/kg/min': (0.0,1.5),       # usual max doses
    'dobutamine_µg/kg/min': (0.0, 20.0),
    'vasopressin_IE/h': (0.0, 6.0),
    'fluids_ml': (0.0, 3000.0),                  # still generous but tighter than artifact range
    'colloids_ml': (0.0, 400.0),
    'fibrinogen_mg/dl': (50.0, 1500.0),
    'platelet_count_G/l': (30.0, 1500.0),
    'lactate_mmol/l': (0.2, 20.0),
    'base_excess_mmol/l': (-20.0, 20.0),
    'urine_output_ml': (0.0, 400.0),
    'thorax-drain_ml': (0.0, 400.0),
    'jackson-pratt_ml': (0.0, 400.0),
    'redon-drain_ml': (0.0,400.0),
    'easy-flow_ml': (0.0, 400.0),
    'robinson-drain_ml': (0.0, 400.0),
    'harnk_ml': (0.0, 2000.0),
    'age': (17.0, 110.0),                         # if you’re only modelling adults
    'heart_rate': (20.0, 220.0),
    'respiratory_rate': (1.0, 70.0),
    'blood_input': (0.0, 2000.0),
    'spo2': (70.0, 100.0),
    'combined_vaso' : (0.0, 3.0),

    'blood_pressure_systolic_mmHg': (60.0, 220.0),
    'blood_pressure_diastolic_mmHg': (30.0, 140.0),
    'blood_pressure_mean_mmHg': (40.0, 160.0),

    'hemoglobin_g/dl': (5.0, 20.0)
}


for col, (min_clamp, max_clamp) in clamp_ranges.items():
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].clip(lower=min_clamp, upper=max_clamp)


In [ ]:


plt.figure(figsize=(10, 5))
sns.histplot(merged_df['blood_pressure_systolic_mmHg'].dropna(), bins=30, kde=True, color='blue')
plt.title(f'Distribution of blood_pressure_systolic_mmHg', fontsize=14)
plt.xlabel('blood_pressure_systolic_mmHg', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.show()




merged_df['drain_sum'] = merged_df['jackson-pratt_ml'] + merged_df['redon-drain_ml'] + merged_df['robinson-drain_ml'] + merged_df['thorax-drain_ml'] + merged_df['easy-flow_ml'] 
merged_df.drop(columns=['jackson-pratt_ml', 'redon-drain_ml', 'robinson-drain_ml', 'thorax-drain_ml','easy-flow_ml'])










In [ ]:
vasos = ['norepinephrine_µg/kg/min', 'vasopressin_IE/h','dobutamine_µg/kg/min']

for vaso in vasos: 
    print(vaso)
    print(merged_df.loc[merged_df[vaso]>0, vaso].describe())
    print('____________________________')

In [ ]:
new_order = [
    'utcChartTime', 'encounterId', 'age', 'sex_or_gender',
    'blood_pressure_diastolic_mmHg', 'blood_pressure_mean_mmHg', 'blood_pressure_systolic_mmHg',
    'combined_vaso', 'colloids_ml', 'fluids_ml', 'fibrinogen_mg/dl',
    'lactate_mmol/l', 'platelet_count_G/l', 'hemoglobin_g/dl',
    'base_excess_mmol/l', 'harnk_ml', 'drain_sum', 'heart_rate', 'respiratory_rate', 'spo2', 'blood_input'
]

# reorder, raising if any are missing
merged_df = merged_df.reindex(columns=new_order)


sns.set_style('whitegrid')


In [ ]:

cols_to_change_to_NaN = ['harnk_ml', 'drain_sum', 'blood_input', 'combined_vaso', 'colloids_ml', 'fluids_ml']
for col in cols_to_change_to_NaN:
    merged_df.loc[merged_df[col]==0,col] = np.NaN

cols_to_save = new_order + ['sex_or_gender'] 
merged_df[cols_to_save].to_csv('data_internal.csv', index=False)
print("Internal data saved.")

In [ ]:
merged_df['hemoglobin_g/dl'].describe()

In [ ]:
merged_df['hemoglobin_g/dl'].value_counts()

In [ ]:
# 1) Sort by encounter and time
merged_df = merged_df.sort_values(['encounterId', 'utcChartTime'])

# 2) Group columns by imputation strategy
med_rate_cols = [
    'combined_vaso'
]
vital_lab_cols = [
    'blood_pressure_systolic_mmHg', 'blood_pressure_mean_mmHg',
    'blood_pressure_diastolic_mmHg', 'spo2', 'fibrinogen_mg/dl', 
    'platelet_count_G/l', 'lactate_mmol/l', 'heart_rate', 'respiratory_rate',
    'base_excess_mmol/l'
]
sum_cols = [
    'fluids_ml', 'colloids_ml', 'harnk_ml', 'drain_sum', 'blood_input'
]

# 3) Fill volumes/sums with 0 (Missing volume = No volume given/output)
merged_df[sum_cols] = merged_df[sum_cols].fillna(0)

# 4) Forward-fill (ffill) within each encounter
# This maintains the last known state (e.g., last BP or last Med rate)
all_continuous = med_rate_cols + vital_lab_cols
merged_df[all_continuous] = (
    merged_df
    .groupby('encounterId', sort=False)[all_continuous]
    .ffill()
)

# 5) Handle remaining initial NaNs (the gaps before the first measurement)

# A: For Meds, initial NaN = 0 (The patient wasn't on the drug yet)
merged_df[med_rate_cols] = merged_df[med_rate_cols].fillna(0)

# B: For Vitals/Labs, initial NaN = Global Median (The patient is alive/average)
vitals_medians = merged_df[vital_lab_cols].median(numeric_only=True)
merged_df[vital_lab_cols] = merged_df[vital_lab_cols].fillna(vitals_medians)

In [ ]:
import os
merged_df.to_parquet(os.path.join('/home/lkapral/hb/data/', f'muw_df.parquet'))

In [ ]:
df = merged_df

In [ ]:
features = med_rate_cols+vital_lab_cols+vital_lab_cols
for col in features:
    
    print(col)
    print(df[col].describe())
    print(df.loc[df[col]>0,col].describe())
    print('___________________________________ \n')
